In [1]:
# Version1_Subset_Sampling_FDR
#  << 데이터 가용성 확보를 위한 경로 리스트 구축 및 데이터셋 규격화 >>
# 소규모 데이터셋 환경에서의 데이터 효율적 AI 모델링 및 학습 속도 최적화를 위해 클래스별 500개 샘플 추출
import os
import random

# 메모리 점유율 최소화를 위해 전체 픽셀 데이터 대신 파일 경로 리스트만 사전 구축
# 학습 시점에 텐서 변환 수행할 예정
def get_sampled_files(directory, sample_size=500):
    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
    all_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            all_files.append(os.path.join(root, file))
    
    # [학습할 데이터량] 
    # 랜덤하게 500개만 섞어서 추출
    # 만약 원래 개수가 500개 이하라면 전체 반환
    if len(all_files) > sample_size:
        sampled_files = random.sample(all_files, sample_size)
    else:
        sampled_files = all_files
        
    return sampled_files

FIGHTER_DIR = '/kaggle/input/datasets/jrmymimran/fighterjets'
DRONE_DIR = '/kaggle/input/datasets/dasmehdixtr/drone-dataset-uav'
ROCKET_DIR = '/kaggle/input/datasets/eneskosar19/rocket-dataset-for-image-detection-labelled'

# 프로토타입 단계에서의 PoC(Proof of Concept) 가속화를 위해 클래스별 균등 서브셋 500개로 구축
fighter_500 = get_sampled_files(FIGHTER_DIR, 500)
drone_500 = get_sampled_files(DRONE_DIR, 500)
rocket_500 = get_sampled_files(ROCKET_DIR, 500)

print(f"추출 완료: 전투기({len(fighter_500)}), 드론({len(drone_500)}), 로켓({len(rocket_500)})")

추출 완료: 전투기(500), 드론(500), 로켓(500)


In [2]:
# Version2_Geometric_Safety_Pipeline 
# <<원본 이미지의 기하학적 형상 보존(Zero-Padding) 및 데이터 무결성 검증(Fail-Safe)이 통합된 딥러닝 전처리 파이프라인>>
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import os

# 클래스 [1] : 원본 이미지 형상을 보존하며 추론 엔진용 텐서로 변환하는 데이터 공급 클래스
class AeroObjectDataset(Dataset):
    def __init__(self, file_paths, labels, img_size=224, is_train=False):
        self.file_paths = file_paths
        self.labels = labels
        self.img_size = img_size
        self.is_train = is_train
        
        # [데이터 증강 및 텐서 변환 파이프라인]
        if self.is_train:
            self.transform = transforms.Compose([
                transforms.RandomHorizontalFlip(),
                # 50% 확률로 수평 반전
                transforms.RandomRotation(15), 
                # 요격 각도 변화 모사 -> 각도를 너무 많이 잡으면 원본과 동 떨어진 오버피팅 우려해 15도로 초기화 
                transforms.ColorJitter(brightness=0.2, contrast=0.2), 
                # 밝기/대비 변화 모사 -> 1.0으로 잡아버리면 원본이미지 훼손이 있어 학습 퀄리티 저해할 것 대비 0.2로 초기화
                transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 3)), 
                # kernel_size(눈의 시야) -> 작은 시야로 세밀하게 볼 5부터 큰 시야로 넓게도 보도록 9까지로 잡음
                # sigma(번짐의 강도) -> 블러가 거의 없는 0.1에서 아주 강력한 번짐의 3까지로 잡음.
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
                # Tesnor : 3차원(층, 줄, 칸) 공간 주소에 배정된 RRB 색상값들을 엔진의 표준 규격에 맞춰 교정
                # Resnet-18의 평균 값 -> [0.485, 0.456, 0.406]
                # Resnet-18의 평균 표준편차 -> [0.229, 0.224, 0.225]
            ])
        else:
            # 검증/테스트 시에는 증강 없이 정규화만 수행
            self.transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])

    # [ 종횡비 유지를 위한 중앙 정렬 제로 패딩 ]
    def _zero_padding(self, image):
        
        w, h = image.size
        max_dim = max(w, h)
        
        # 검은색(0,0,0) 정사각형 배경을 생성하고 원본 이미지를 중앙에 배치하여 종횡비 보존
        new_image = Image.new('RGB', (max_dim, max_dim), (0, 0, 0))
        
        # 원본 이미지를 캔버스 중앙에 배치
        paste_pos = ((max_dim - w) // 2, (max_dim - h) // 2)
        new_image.paste(image, paste_pos)

        # 최종 모델 입력 사이즈(224x224)로 리사이징
        # 이미지 작으면 확대 / 이미지 크면 축소 -> 기존에도 정사각형 형태라 원본에 훼손이 없음
        return new_image.resize((self.img_size, self.img_size))

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        
        # Fail-safe: 비정상 파일 유입 시 예외 처리 및 차단
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"파일 손상으로 무작위 대체: {os.path.basename(img_path)}")
            import random
            random_idx = random.randint(0, len(self.file_paths) - 1)
            return self.__getitem__(random_idx) # 다른 정상 샘플로 대체하여 반환
            
        padded_image = self._zero_padding(image) # 제로 패딩 적용 
        tensor_image = self.transform(padded_image) # 텐서 변환 및 데이터 증강
        
        return tensor_image, torch.tensor(self.labels[idx], dtype=torch.long)


# 클래스 [2] : 데이터 무결성을 검증하고 손상된 샘플을 데이터셋에서 영구 제외하는 모듈
class FailSafeValidator:
    def __init__(self, file_paths, labels):
        self.file_paths = file_paths
        self.labels = labels
        self.clean_paths = []
        self.clean_labels = []
        
    # 전체 리스트를 순회하며 실제 열리는 이미지인지 검증 (학습 시작 전 동작)
    def filter_bad_images(self):
        print(f"[*] 데이터 무결성 검사 시작 (대상: {len(self.file_paths)}개)...")

        removed_count = 0
        
        for path, label in zip(self.file_paths, self.labels):
            try:
                # 1. 파일 존재 여부 확인
                if not os.path.exists(path):
                    raise FileNotFoundError
                
                # 2. 실제로 이미지가 열리고 RGB 변환이 가능한지 확인
                with Image.open(path) as img:
                    img.verify() # 파일 구조 1차 검증
                with Image.open(path) as img:
                    img.load() # 실제 픽셀 데이터 손상 여부 2차 검증
                
                # 검증 통과 시 '깨끗한 리스트'에 추가
                self.clean_paths.append(path)
                self.clean_labels.append(label)
                
            except Exception as e:
                removed_count += 1 
                
                # 국방 시스템 로그 기록 모사 (100 단위)
                if removed_count % 100 == 0 :
                    print(f"무결성 검증 실패 - 제외 처리됨: {os.path.basename(path)} (누적 {removed_count}개 제외)...")
                continue

        print(f"검사 완료 : {removed_count}개의 불량 샘플 제거됨.")
        return self.clean_paths, self.clean_labels

In [3]:
# Version3_ResNet18_Architecture_Trainer
# << 항공 객체 식별용 다차원 특징 추출 엔진(ResNet-18) 설계 및 지도학습 최적화 모듈 >>
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim

# 클래스 [3] : ResNet-18을 활용하여 추출한 데이터셋의 다차원 특징을 추출하는 신경망 클래스
class AeroObjectClassifier(nn.Module): # nn.Module 상속 -> 파이토치에서 모든 신경망 모델이 가져야할 기본 뼈대를 받음
    def __init__(self, num_classes=3, pretrained=True):
        super().__init__()
        # ResNet-18을 선택한 이유 
        # -> 이미지의 선,면 및 구체적인 형태를 추출하는데 매우 탁월한 성능을 가짐
        # -> 1500개라는 소량의 데이터에 적합하다고 판단
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        self.backbone = models.resnet18(weights=weights)
        
        # fc -> 이미지 특징들을 하나로 모아 최종적인 결과를 도출하는 출력층
        # in_features 통해 모델에 적합한 규격에 맞춤 (3개의 데이터셋(클래스))
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, num_classes)

        # ONNX 추출 시에만 확률을 반환하도록 제어하는 스위치
        self.export_mode = False
        
    def forward(self, x):
        logits = self.backbone(x)
        # 1. 형상 보존형 전처리(Zero-Padding)를 마친 텐서 x를 입력받음 
        # 2. CNN 필터를 통해 객체의 기하학적 특징과 실루엣을 추출하여 연산 수행 
        # 3. 각 클래스(전투기, 드론, 로켓)에 대한 예측 점수(Logits)가 담긴 텐서 반환

        if self.export_mode:
            return torch.softmax(logits, dim=1)

        return logits


# 클래스 [4] : 모델을 학습시키고 가중치를 추출하는 클래스
class ModelTrainer:
    def __init__(self, model: nn.Module, device: str, learning_rate=0.001):
        self.model = model # 외부에서 생성한 AI 모델 객체를 클래스 내부 멤버 변수로 할당
        self.device = torch.device(device) # 연산을 수행할 물리적 장치(CPU 또는 GPU)를 파이토치 엔진 규격에 맞게 설정
        self.model.to(self.device) # 모델의 모든 가중치와 신경망 데이터를 지정된 장치의 메모리로 전송하여 실행 준비를 마침
        
        # 오발사 방지를 위한 엄격한 지도학습(CrossEntropyLoss) 채택
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)

    def train_epoch(self, train_loader):
        self.model.train() # 학습 모드 활성화 
        running_loss = 0.0 # 하나의 epoch가 처리되면 다시 0으로 초기화
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(self.device), labels.to(self.device)

            # 국방 시스템의 오발사 방지를 위해, 시행착오 기반의 강화학습(RL) 대신 판독 신뢰성이 확보된 지도학습(SL) 채택
            self.optimizer.zero_grad()    # 1. 기울기 초기화
            outputs = self.model(inputs)  # 2. 순전파 (forward)
            loss = self.criterion(outputs, labels) # 3. 오차 계산
            loss.backward()               # 4. 역전파 (backward)
            self.optimizer.step()         # 5. 가중치 업데이트

            # 순수 오차 수치만 누적함 (학습용)
            running_loss += loss.item()

        # Epoch의 최종 평균 오차를 계산해서 반환하는 동작
        return running_loss / len(train_loader)

    def evaluate(self, val_loader):
        self.model.eval() # 검증 모드 (가중치 업데이트 중단)
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad(): # 기울기 계산 비활성화 (메모리 절약 및 속도 향상)
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(self.device), labels.to(self.device)
                outputs = self.model(inputs)
                
                loss = self.criterion(outputs, labels)
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1) # 가장 높은 확률의 클래스 추출
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
        accuracy = 100 * correct / total
        return val_loss / len(val_loader), accuracy

In [4]:
# Version4_ReasoningGuard_Core_Engine_Generation
# << Reasoning Guard 실전 파이프라인 총괄 컨트롤러 및 실행 모듈 >>
# Scikit-learn(딥러닝X, 고전적이지만 회귀, 클러스트링 같은 강력한 AL 포함 -> 빠름) 등이 존재

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import time
import torch

!pip install onnx onnxscript
import torch.onnx

# 클래스 [5] : 데이터 준비부터 학습, ONNX 추출까지의 전체 파이프라인을 통제하는 컨트롤러 클래스
class ReasoningGuardPipeline:
    # 파이프라인 가동되기 전, 필요한 기본 설정값과 자원을 할당받는 준비 단계
    def __init__(self, all_paths, all_labels, batch_size=32, num_epochs=10):
        self.all_paths = all_paths
        self.all_labels = all_labels
        self.batch_size = batch_size
        self.num_epochs = num_epochs
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
        # 내부에서 사용할 객체들 초기화 대기
        self.train_loader = None
        self.val_loader = None
        self.model = None
        self.trainer = None

    # 전처리 및 데이터 로드 단계
    def prepare_data(self):
        print("\n[Phase 1] 데이터 준비 및 Fail-safe 검증 가동")
        
         # FailSafeValidator를 호출해 깨지거나 손상된 이미지 원천 차단
        validator = FailSafeValidator(self.all_paths, self.all_labels)
        clean_paths, clean_labels = validator.filter_bad_images()

        # 전체 데이터를 학습용과 검증용 (8:2)로 나눔
        train_paths, val_paths, train_labels, val_labels = train_test_split(
            clean_paths, clean_labels, test_size=0.2, stratify=clean_labels, random_state=42
        )

        # Train (80%) / Validation (20%) Dataset 생성
        # val_dataset은 is_train=False를 통해 데이터 증강 가동하지 않음 -> 검증 및 실전 모드
        train_dataset = AeroObjectDataset(train_paths, train_labels, img_size=224, is_train=True)
        val_dataset = AeroObjectDataset(val_paths, val_labels, img_size=224, is_train=False)

        print(f"[*] 분할 완료 -> Train: {len(train_dataset)}개 / Validation: {len(val_dataset)}개")

        # DataLoader 설정
        self.train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=2, drop_last=True)
        self.val_loader = DataLoader(val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=2)

    # 준비된 데이터를 바탕으로 AI모델(ResNet-18)을 실제 전장 환경에 맞게 훈련시키는 핵심 엔진 가동 단계
    def execute_training(self):
        print(f"\n[Phase 2] 전술 객체 식별 AI 훈련 개시 (총 {self.num_epochs} Epochs)")
        
        # 연산 장치 및 추론 엔진 초기화
        print(f"[*] 작전 통제소(학습 장치) 설정 완료: {self.device.upper()} 가동")
        self.model = AeroObjectClassifier(num_classes=3, pretrained=True)
        self.trainer = ModelTrainer(model=self.model, device=self.device, learning_rate=0.001)

        start_time = time.time()

        # 본격적인 학습 및 검증 루프
        for epoch in range(self.num_epochs):
            epoch_start = time.time()
            # epoch가 시작된 시간을 기록

            # [훈련 모드] -> 평균 오차 점수 저장 (낮을수록 좋음)
            train_loss = self.trainer.train_epoch(self.train_loader)
            # [검증 모드] -> 시험의 오차, 정답률 저장
            val_loss, val_accuracy = self.trainer.evaluate(self.val_loader)

            # 현재 시간 - 기록해둔 시간 -> 1번의 epoch 수행하는데 걸린시간 도출
            epoch_duration = time.time() - epoch_start
            print(f"Epoch [{epoch+1}/{self.num_epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.2f}% | 소요시간: {epoch_duration:.2f}s")

        total_time = time.time() - start_time
        print(f"\n[!] 전체 훈련 상황 종료! 총 소요 시간: {total_time/60:.2f}분")

        # 학습된 가중치 저장 (ONNX 변환 준비)
        save_path = "reasoning_guard_poc.pth" # .pth 파이썬 확장자로 파일이름과 경로 저장
        
        torch.save(self.model.state_dict(), save_path)
        # 파이토치 저장 기능 -> 현재 모델이 가지고 있는 모든 가중치와 편향 값들을 지정한 경로의 파일로 하드디스크에 기록
        # save_path 파일을 통해 나중 학습을 생략하고 C++ 연동을 위한 ONNX 추출 작업 바로 수행 가능
        
        print(f"[*] AI 가중치 저장 완료: {save_path}")

    def export_to_onnx(self, output_filename="reasoning_guard_engine.onnx"):
        print("\n[Phase 3] 하이브리드 아키텍처 연동을 위한 ONNX 엔진 추출")
        self.model.eval() # 추론 모드로 변경
        
        # 방금 만든 모델 내부의 Softmax 스위치를 여기서 킴
        self.model.export_mode = True 
        
        dummy_input = torch.randn(1, 3, 224, 224).to(self.device)
        # torch.randn(한번에 처리할 이미지 개수, RGB컬러이미지, 이미지의 가로세로 해상도)
        
        torch.onnx.export(
            self.model, dummy_input, output_filename,
            # 모델 구조와 가중치 기록 위한 필수 3요소
            
            export_params=True, opset_version=18, do_constant_folding=True,
            # export_params : 학습된 가중치
            # opset_version : 버전 규격
            # do_constant_folding : 모델 내부 수식중 미리 계산할 수 있는 상수부분 최적화 -> 추론속도 높임
            
            # 출력 이름표를 'logits'에서 'probabilities'로 변경
            input_names=['input_tensor'], output_names=['shoot_hold_probabilities']
        )
        print(f"[!] 성공: C++ 연동용 ONNX 파일 생성 완료 -> {output_filename}")


# ========================================
# 실제 컨트롤러 가동 (모든 파이프라인이 실행)
# ========================================
if __name__ == "__main__": # 사용자가 직접 이 파일을 실행했을때만 코드 동작하도록 제안함

    all_paths = fighter_500 + drone_500 + rocket_500
    all_labels = [0] * len(fighter_500) + [1] * len(drone_500) + [2] * len(rocket_500)
    
    # 파일 경로와 라벨 리스트 병합 (상단에서 이미 선언된 데이터 사용)
    pipeline = ReasoningGuardPipeline(all_paths, all_labels, batch_size=32, num_epochs=10)
    pipeline.prepare_data() # 데이터 준비하고
    pipeline.execute_training() # 훈련시키고
    pipeline.export_to_onnx() # onnx로 추출

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 11.5 MB/s eta 0:00:00

[Phase 1] 데이터 준비 및 Fail-safe 검증 가동
[*] 데이터 무결성 검사 시작 (대상: 1500개)...
무결성 검증 실패 - 제외 처리됨: pic_1036.xml (누적 100개 제외)...
무결성 검증 실패 - 제외 처리됨: pic_537.txt (누적 200개 제외)...
무결성 검증 실패 - 제외 처리됨: 076fa5e65495a3b9.txt (누적 300개 제외)...
무결성 검증 실패 - 제외 처리됨: 35edeb4d31703350.txt (누적 400개 제외)...
무결성 검증 실패 - 제외 처리됨: 61f24a600b42b5d3.txt (누적 500개 제외)...
검사 완료 : 501개의 불량 샘플 제거됨.
[*] 분할 완료 -> Train: 799개 / Validation: 200개

[Phase 2] 전술 객체 식별 AI 훈련 개시 (총 10 Epochs)
[*] 작전 통제소(학습 장치) 설정 완료: CUDA 가동
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 220MB/s]
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [1/10] | Train Loss: 0.6261 | Val Loss: 2.6237 | Val Acc: 45.00% | 소요시간: 18.40s


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [2/10] | Train Loss: 0.3836 | Val Loss: 0.4762 | Val Acc: 85.00% | 소요시간: 18.27s


...

Epoch [9/10] | Train Loss: 0.1431 | Val Loss: 0.2470 | Val Acc: 91.50% | 소요시간: 18.39s


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [10/10] | Train Loss: 0.0919 | Val Loss: 0.4977 | Val Acc: 86.50% | 소요시간: 18.24s

[!] 전체 훈련 상황 종료! 총 소요 시간: 3.09분
[*] AI 가중치 저장 완료: reasoning_guard_poc.pth

[Phase 3] 하이브리드 아키텍처 연동을 위한 ONNX 엔진 추출


W0429 13:07:24.384000 57 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0429 13:07:24.386000 57 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0429 13:07:24.388000 57 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0429 13:07:24.390000 57 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `AeroObjectClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `AeroObjectClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[!] 성공: C++ 연동용 ONNX 파일 생성 완료 -> reasoning_guard_engine.onnx


In [5]:
# 훈련 끝난 모델을 ONNX로 변환 수행
pipeline.export_to_onnx()


[Phase 3] 하이브리드 아키텍처 연동을 위한 ONNX 엔진 추출


W0429 13:07:27.903000 57 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0429 13:07:27.905000 57 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0429 13:07:27.907000 57 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0429 13:07:27.909000 57 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `AeroObjectClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `AeroObjectClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[!] 성공: C++ 연동용 ONNX 파일 생성 완료 -> reasoning_guard_engine.onnx
